# Enrich Sample Ad Copy Dataset
Create synthetic user data (interactions, views, etc.) to match the desired format (meta_ads_template). Hugging face ad copy data only has text + shape (see 01_exploration), so we need to generate:
- ad_id           str
- campaign_id     (optional)
- headline_text   str
- body_text       str
- call_to_action  str
- platform        str 
- placement       float64
- ad_format       str
- category        str
- impressions     int64
- clicks          int64  
- conversions     int64  
- spend           float64

Afterwards, we can run compute_metrics again to enrich dataset.

First take a look at desired template:

In [ ]:
# Sample of template
import pandas as pd
template = pd.read_csv('../data/pipeline/meta_ads_template.csv')
template.head(5)

,ad_id,campaign_id,headline_text,body_text,call_to_action,platform,placement,ad_format,category,impressions,clicks,conversions,spend
0,meta_ad_001,NaN,Save 20% on Running Shoes Today,Limited-time offer on our best-selling running...,Shop Now,facebook,NaN,image,ecommerce,58210,913,124,834.71
1,meta_ad_002,NaN,Organize Your Work in One Place,Our project management platform helps remote t...,Start Free Trial,instagram,NaN,image,saas,42135,214,31,289.50
2,meta_ad_003,NaN,Master Data Analytics in 8 Weeks,Join our intensive online bootcamp and build a...,Apply Now,facebook,NaN,video,education,30112,452,46,512.39
3,meta_ad_004,NaN,Fresh Deals on Home Office Gear,"Upgrade your desk setup with ergonomic chairs,...",Shop Deals,facebook,NaN,image,ecommerce,76544,1023,138,954.80
4,meta_ad_005,NaN,The CRM Built for Small Teams,"Track leads, automate follow-ups, and close mo...",Book Demo,instagram,NaN,video,saas,38950,198,22,314.27


Compared to current csv:

In [ ]:
df = pd.read_csv('../data/sample_ads.csv')
df.head(5)

,text,dimensions
0,AGF\r\nAMAZING GRACE FELLOWSHIP\r\n1061 EASTLA...,"(300, 250)"
1,NO\r\nZOLUCKY\r\n- UP TO -\r\n50%\r\nshop now,"(728, 90)"
2,Academy\r\nSPORTS+OUTDOORS\r\nPRESENTS\r\n::HA...,"(160, 600)"
3,TIME TO\r\nOREGON'S\r\nMT.HOOD\r\nTERRITORY\r\...,"(728, 90)"
4,FARMERS\r\nINSURANCE\r\nJeremy Belcourt\r\nCON...,"(728, 90)"


## Pipeline to enrich dataset via LLM in Ollama
Using LLM, I'll convert the current df to match the format of the template csv. We'll need to categorize, split text, and generate numbers/tags. This pipeline can later be converted to a util function if useful.

In [21]:
# Import required libraries for LangChain + Ollama pipeline
import pandas as pd
import time
import random
import numpy as np
from typing import Dict, List, Optional
import re

# LangChain imports
from langchain_ollama import OllamaLLM
from langchain.prompts import PromptTemplate
from langchain.output_parsers import PydanticOutputParser
from langchain.schema import OutputParserException
from pydantic import BaseModel, Field

# Define output schema for structured parsing
class AdComponents(BaseModel):
    headline_text: str = Field(description="Main catchy title or primary message (1-2 lines max)")
    body_text: str = Field(description="Detailed description or supporting text")
    call_to_action: str = Field(description="Action phrase like 'Shop Now', 'Learn More', etc.")
    category: str = Field(description="Business category: ecommerce, saas, education, healthcare, finance, entertainment, travel, food, automotive, or real_estate")
    platform: str = Field(description="Suggested platform: 'facebook' or 'instagram'")

# Initialize LangChain Ollama
MODEL_NAME = "qwen3:8b"  # Adjust based on your installed model
llm = OllamaLLM(model=MODEL_NAME, temperature=0.3)

# Create output parser
parser = PydanticOutputParser(pydantic_object=AdComponents)

# Create prompt template
prompt_template = PromptTemplate(
    template="""Analyze the following advertisement text and extract these components:

{format_instructions}

Advertisement text:
{ad_text}

Please provide a structured response:""",
    input_variables=["ad_text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

# Create the chain
analysis_chain = prompt_template | llm | parser

# Test LangChain connection
print("Testing LangChain + Ollama connection...")
try:
    test_ad = "Limited time offer! Save 50% on premium running shoes. Free shipping worldwide. Shop now and upgrade your fitness game!"
    test_result = analysis_chain.invoke({"ad_text": test_ad})
    print(f"LangChain test successful!")
    print(f"Test result: {test_result}")
except Exception as e:
    print(f"LangChain test failed: {str(e)}")
    print("Make sure Ollama is running and the model is installed.")

Testing LangChain + Ollama connection...
LangChain test successful!
Test result: headline_text='Limited time offer!' body_text='Save 50% on premium running shoes. Free shipping worldwide. Upgrade your fitness game!' call_to_action='Shop now' category='ecommerce' platform='facebook'


In [15]:
def parse_ad_text_langchain(ad_text: str, max_retries: int = 3) -> Dict[str, str]:
    """
    Use LangChain + Ollama to parse raw ad text into structured components
    """
    for attempt in range(max_retries):
        try:
            # Use the LangChain chain
            result = analysis_chain.invoke({"ad_text": ad_text})
            
            # Convert Pydantic model to dict
            return {
                'headline_text': result.headline_text,
                'body_text': result.body_text,
                'call_to_action': result.call_to_action,
                'category': result.category.lower(),
                'platform': result.platform.lower()
            }
            
        except OutputParserException as e:
            print(f"Parser error on attempt {attempt + 1}: {str(e)}")
            # Try with a simpler prompt on retry
            if attempt < max_retries - 1:
                time.sleep(1)
                continue
            else:
                # Fallback to default values
                return {
                    'headline_text': ad_text[:100],  # Use first 100 chars as headline
                    'body_text': ad_text[100:400],   # Rest as body
                    'call_to_action': 'Learn More',
                    'category': 'ecommerce',
                    'platform': 'facebook'
                }
        except Exception as e:
            print(f"Unexpected error on attempt {attempt + 1}: {str(e)}")
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                # Fallback to default values
                return {
                    'headline_text': ad_text[:100],
                    'body_text': ad_text[100:400],
                    'call_to_action': 'Learn More',
                    'category': 'ecommerce',
                    'platform': 'facebook'
                }
    
    # This shouldn't be reached, but just in case
    return {
        'headline_text': ad_text[:100],
        'body_text': ad_text[100:400],
        'call_to_action': 'Learn More',
        'category': 'ecommerce',
        'platform': 'facebook'
    }

def determine_ad_format(dimensions: str, text: str) -> str:
    """
    Determine ad format based on dimensions and content analysis
    """
    try:
        # Parse dimensions
        dims = dimensions.strip('()').split(',')
        width = int(dims[0].strip())
        height = int(dims[1].strip())
        
        # Video indicators in text
        video_keywords = ['video', 'watch', 'play', 'stream', 'tutorial', 'demo', 'webinar']
        has_video_content = any(keyword in text.lower() for keyword in video_keywords)
        
        # Determine format based on dimensions and content
        if has_video_content or (width >= 400 and height >= 300):
            return 'video'
        else:
            return 'image'
            
    except:
        return 'image'  # default

def generate_synthetic_metrics(category: str, ad_format: str) -> Dict[str, float]:
    """
    Generate realistic synthetic metrics based on category and format
    """
    # Base metrics by category (industry benchmarks)
    category_metrics = {
        'ecommerce': {'base_ctr': 0.015, 'base_cvr': 0.08, 'base_cpm': 12.0},
        'saas': {'base_ctr': 0.012, 'base_cvr': 0.05, 'base_cpm': 15.0},
        'education': {'base_ctr': 0.018, 'base_cvr': 0.12, 'base_cpm': 8.0},
        'healthcare': {'base_ctr': 0.014, 'base_cvr': 0.06, 'base_cpm': 18.0},
        'finance': {'base_ctr': 0.010, 'base_cvr': 0.04, 'base_cpm': 22.0},
        'entertainment': {'base_ctr': 0.020, 'base_cvr': 0.15, 'base_cpm': 6.0},
        'travel': {'base_ctr': 0.016, 'base_cvr': 0.07, 'base_cpm': 14.0},
        'food': {'base_ctr': 0.019, 'base_cvr': 0.10, 'base_cpm': 10.0},
        'automotive': {'base_ctr': 0.013, 'base_cvr': 0.03, 'base_cpm': 16.0},
        'real_estate': {'base_ctr': 0.011, 'base_cvr': 0.02, 'base_cpm': 20.0}
    }
    
    # Get base metrics or use ecommerce as default
    base = category_metrics.get(category, category_metrics['ecommerce'])
    
    # Video ads typically perform better
    format_multiplier = 1.3 if ad_format == 'video' else 1.0
    
    # Generate impressions (20K to 100K range)
    impressions = random.randint(20000, 100000)
    
    # Calculate clicks based on CTR with some variance
    ctr = base['base_ctr'] * format_multiplier * random.uniform(0.7, 1.4)
    clicks = int(impressions * ctr)
    
    # Calculate conversions based on CVR with some variance
    cvr = base['base_cvr'] * random.uniform(0.6, 1.3)
    conversions = int(clicks * cvr)
    
    # Calculate spend based on CPM with some variance
    cpm = base['base_cpm'] * random.uniform(0.8, 1.5)
    spend = round((impressions / 1000) * cpm, 2)
    
    return {
        'impressions': impressions,
        'clicks': clicks,
        'conversions': conversions,
        'spend': spend
    }

# Test the improved functions with a sample
print("Testing LangChain-based ad text parsing...")
sample_text = df.iloc[0]['text']
print(f"Sample text: {sample_text[:100]}...")
parsed = parse_ad_text_langchain(sample_text)
print(f"Parsed result: {parsed}")

Testing LangChain-based ad text parsing...
Sample text: AGF
AMAZING GRACE FELLOWSHIP
1061 EASTLAND DRIVE NORTH
TWIN FALLS IDAHO
Service times: Sundays a...
Parsed result: {'headline_text': 'AMAZING GRACE FELLOWSHIP', 'body_text': '1061 EASTLAND DRIVE NORTH, TWIN FALLS IDAHO. Service times: Sundays at 8:30 and 10:30 AM.', 'call_to_action': 'Go to agf.org for more information', 'category': 'education', 'platform': 'none'}
Parsed result: {'headline_text': 'AMAZING GRACE FELLOWSHIP', 'body_text': '1061 EASTLAND DRIVE NORTH, TWIN FALLS IDAHO. Service times: Sundays at 8:30 and 10:30 AM.', 'call_to_action': 'Go to agf.org for more information', 'category': 'education', 'platform': 'none'}


### Takes quite a while. May need to choose a smaller LLM for this step:

## Performance Optimization for Large Datasets

For processing 7,000 ads efficiently, consider these optimizations:

In [22]:
# Model Performance Configuration for 7K Ads
# Choose your model based on speed vs quality tradeoff

FAST_MODELS = {
    "gemma:2b": {"speed": "⚡⚡⚡", "time_estimate": "1-2 hours", "quality": "Good"},
    "phi3:mini": {"speed": "⚡⚡", "time_estimate": "2-4 hours", "quality": "Very Good"}, 
    "qwen3:8b": {"speed": "⚡", "time_estimate": "4-8 hours", "quality": "Excellent"}
}

# Switch model here for faster processing
SELECTED_MODEL = "phi4-mini"  # Change to "phi3:mini" or "gemma:2b" for speed

# Performance optimizations
OPTIMIZED_CONFIG = {
    "batch_size": 50,          # Larger batches for efficiency
    "delay_between_ads": 0.1,  # Reduced delay (from 0.3s)
    "batch_pause": 2,          # Shorter breaks (from 3s)
    "max_retries": 2,          # Fewer retries (from 3)
    "temperature": 0.1         # Lower temperature for faster processing
}

print(f"Selected Model: {SELECTED_MODEL}")

# Reinitialize with optimized settings
llm_optimized = OllamaLLM(
    model=SELECTED_MODEL, 
    temperature=OPTIMIZED_CONFIG["temperature"]
)

# Create optimized chain
analysis_chain_optimized = prompt_template | llm_optimized | parser

Selected Model: phi4-mini


In [25]:
# Optimized parsing function for high-volume processing
def parse_ad_text_optimized(ad_text: str) -> Dict[str, str]:
    """
    Optimized version for processing large datasets
    """
    max_retries = OPTIMIZED_CONFIG["max_retries"]
    
    for attempt in range(max_retries):
        try:
            # Use optimized chain
            result = analysis_chain_optimized.invoke({"ad_text": ad_text})
            
            return {
                'headline_text': result.headline_text,
                'body_text': result.body_text,
                'call_to_action': result.call_to_action,
                'category': result.category.lower(),
                'platform': result.platform.lower()
            }
            
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(0.5)  # Shorter retry delay
                continue
            else:
                # Fast fallback - no complex processing
                lines = ad_text.split('\n')
                return {
                    'headline_text': lines[0][:100] if lines else ad_text[:100],
                    'body_text': ' '.join(lines[1:])[:300] if len(lines) > 1 else ad_text[100:400],
                    'call_to_action': 'Learn More',
                    'category': 'ecommerce',
                    'platform': 'facebook'
                }
    
    # Fallback
    return {
        'headline_text': ad_text[:100],
        'body_text': ad_text[100:400],
        'call_to_action': 'Learn More',
        'category': 'ecommerce',
        'platform': 'facebook'
    }

# Optimized batch processing function
def process_large_dataset_optimized(df: pd.DataFrame, start_idx: int = 0, max_ads: int = 7000) -> pd.DataFrame:
    """
    Optimized processing for large datasets with progress tracking
    """
    enriched_ads = []
    end_idx = min(start_idx + max_ads, len(df))
    
    batch_size = OPTIMIZED_CONFIG["batch_size"]
    delay = OPTIMIZED_CONFIG["delay_between_ads"]
    batch_pause = OPTIMIZED_CONFIG["batch_pause"]
    
    print(f"🚀 Processing {end_idx - start_idx} ads with optimized settings...")
    print(f"📊 Batch size: {batch_size}, Model: {SELECTED_MODEL}")
    
    import time as time_module
    start_time = time_module.time()
    
    for i in range(start_idx, end_idx):
        try:
            # Progress tracking - show progress every few ads
            ads_processed = i - start_idx + 1
            total_ads = end_idx - start_idx
            
            # Show progress on first ad, then every 5 ads for small batches, every 25 for large batches
            progress_interval = 5 if total_ads <= 50 else 25
            
            if (ads_processed == 1 or 
                ads_processed % progress_interval == 0 or 
                i == end_idx - 1):
                
                elapsed = time_module.time() - start_time
                progress = ads_processed / total_ads * 100
                
                if ads_processed > 0 and elapsed > 0:
                    avg_time_per_ad = elapsed / ads_processed
                    remaining_ads = total_ads - ads_processed
                    eta_seconds = remaining_ads * avg_time_per_ad
                    eta_minutes = eta_seconds / 60
                    
                    print(f"📊 Progress: {progress:.1f}% ({ads_processed}/{total_ads}) | "
                          f"⏱️  Avg: {avg_time_per_ad:.2f}s/ad | 🕒 ETA: {eta_minutes:.1f} min")
            
            # Batch pause
            if (i - start_idx) % batch_size == 0 and i > start_idx:
                print(f"⏸️  Batch complete. Taking {batch_pause}s break...")
                time.sleep(batch_pause)
            
            # Process single ad
            row = df.iloc[i]
            parsed_components = parse_ad_text_optimized(row['text'])
            ad_format = determine_ad_format(row['dimensions'], row['text'])
            metrics = generate_synthetic_metrics(parsed_components['category'], ad_format)
            
            enriched_ad = {
                'ad_id': f"enriched_ad_{i + 1:04d}",
                'campaign_id': f"campaign_{random.randint(1, 50):03d}",
                'headline_text': parsed_components['headline_text'][:100],
                'body_text': parsed_components['body_text'][:300],
                'call_to_action': parsed_components['call_to_action'],
                'platform': parsed_components['platform'],
                'placement': '',
                'ad_format': ad_format,
                'category': parsed_components['category'],
                'impressions': metrics['impressions'],
                'clicks': metrics['clicks'],
                'conversions': metrics['conversions'],
                'spend': metrics['spend']
            }
            
            enriched_ads.append(enriched_ad)
            
            # Short delay between ads
            time.sleep(delay)
            
        except Exception as e:
            print(f"❌ Error processing ad {i + 1}: {str(e)}")
            continue
    
    total_time = time_module.time() - start_time
    print(f"✅ Completed! Processed {len(enriched_ads)} ads in {total_time/60:.1f} minutes")
    print(f"📈 Average time per ad: {total_time/len(enriched_ads):.2f} seconds")
    
    return pd.DataFrame(enriched_ads)

# Test with a small batch first
print("Testing optimized pipeline with 10 ads...")
test_optimized = process_large_dataset_optimized(df, start_idx=0, max_ads=10)

Testing optimized pipeline with 10 ads...
🚀 Processing 10 ads with optimized settings...
📊 Batch size: 50, Model: phi4-mini
📊 Progress: 50.0% (5/10) | ⏱️  Avg: 1.60s/ad | 🕒 ETA: 0.1 min
📊 Progress: 50.0% (5/10) | ⏱️  Avg: 1.60s/ad | 🕒 ETA: 0.1 min
📊 Progress: 100.0% (10/10) | ⏱️  Avg: 1.71s/ad | 🕒 ETA: 0.0 min
📊 Progress: 100.0% (10/10) | ⏱️  Avg: 1.71s/ad | 🕒 ETA: 0.0 min
✅ Completed! Processed 10 ads in 0.3 minutes
📈 Average time per ad: 1.94 seconds
✅ Completed! Processed 10 ads in 0.3 minutes
📈 Average time per ad: 1.94 seconds


## Time Estimates for 7,000 Ads

| Model | Size | Speed per Ad | Total Time | Quality |
|-------|------|-------------|------------|---------|
| **gemma:2b** | 2B | ~0.8s | **1.5-2 hours** | Good ⭐⭐⭐ |
| **phi3:mini** | 3.8B | ~1.5s | **3-4 hours** | Very Good ⭐⭐⭐⭐ |
| **qwen3:8b** | 8B | ~2.5s | **5-6 hours** | Excellent ⭐⭐⭐⭐⭐ |

### **Recommendation for 7K Ads:**
- **If speed is critical**: Use `gemma:2b` (1-2 hours)
- **Best balance**: Use `phi3:mini` (3-4 hours, very good quality)  
- **Best quality**: Keep `qwen3:8b` (5-6 hours, but run overnight)

### **Optimizations Applied:**
- ✅ Reduced delays between requests
- ✅ Larger batch sizes  
- ✅ Fewer retries on errors
- ✅ Progress tracking with ETA
- ✅ Fast fallback processing

In [ ]:
# Full 7K processing (uncomment when ready)
# WARNING: This will take several hours depending on your model choice

# Option 1: Process all 7,000 ads
# full_dataset_7k = process_large_dataset_optimized(df, start_idx=0, max_ads=7000)

# Option 2: Process in chunks and save incrementally (safer for large datasets)
def process_in_chunks(df, chunk_size=1000, total_ads=7000):
    """
    Process in chunks and save each chunk to avoid losing progress
    """
    all_chunks = []
    
    for chunk_start in range(0, total_ads, chunk_size):
        chunk_end = min(chunk_start + chunk_size, total_ads)
        print(f"\n🔄 Processing chunk {chunk_start//chunk_size + 1}: ads {chunk_start+1} to {chunk_end}")
        
        chunk_df = process_large_dataset_optimized(df, start_idx=chunk_start, max_ads=chunk_size)
        
        # Save each chunk
        chunk_file = f'../data/enriched_ads_chunk_{chunk_start//chunk_size + 1:02d}.csv'
        chunk_df.to_csv(chunk_file, index=False)
        print(f"💾 Saved chunk to {chunk_file}")
        
        all_chunks.append(chunk_df)
        
        # Brief pause between chunks
        time.sleep(5)
    
    # Combine all chunks
    final_df = pd.concat(all_chunks, ignore_index=True)
    final_file = '../data/pipeline/enriched_ads_7k_complete.csv'
    final_df.to_csv(final_file, index=False)
    print(f"\n🎉 All done! Saved complete dataset to {final_file}")
    
    return final_df

# Uncomment to process all 7K ads in chunks:
final_dataset = process_in_chunks(df, chunk_size=1000, total_ads=7000)

print("Ready to process 7,000 ads!")
print("Uncomment the line above to start full processing.")
print(f"Current model: {SELECTED_MODEL}")
print("To switch models, change SELECTED_MODEL in the cell above and re-run.")


🔄 Processing chunk 1: ads 1 to 1000
🚀 Processing 1000 ads with optimized settings...
📊 Batch size: 50, Model: phi4-mini
📊 Progress: 2.5% (25/1000) | ⏱️  Avg: 1.78s/ad | 🕒 ETA: 28.9 min
📊 Progress: 2.5% (25/1000) | ⏱️  Avg: 1.78s/ad | 🕒 ETA: 28.9 min
📊 Progress: 5.0% (50/1000) | ⏱️  Avg: 1.78s/ad | 🕒 ETA: 28.2 min
📊 Progress: 5.0% (50/1000) | ⏱️  Avg: 1.78s/ad | 🕒 ETA: 28.2 min
⏸️  Batch complete. Taking 2s break...
⏸️  Batch complete. Taking 2s break...
📊 Progress: 7.5% (75/1000) | ⏱️  Avg: 1.83s/ad | 🕒 ETA: 28.1 min
📊 Progress: 7.5% (75/1000) | ⏱️  Avg: 1.83s/ad | 🕒 ETA: 28.1 min
📊 Progress: 10.0% (100/1000) | ⏱️  Avg: 1.84s/ad | 🕒 ETA: 27.6 min
📊 Progress: 10.0% (100/1000) | ⏱️  Avg: 1.84s/ad | 🕒 ETA: 27.6 min
⏸️  Batch complete. Taking 2s break...
⏸️  Batch complete. Taking 2s break...
📊 Progress: 12.5% (125/1000) | ⏱️  Avg: 1.88s/ad | 🕒 ETA: 27.5 min
📊 Progress: 12.5% (125/1000) | ⏱️  Avg: 1.88s/ad | 🕒 ETA: 27.5 min
📊 Progress: 15.0% (150/1000) | ⏱️  Avg: 1.89s/ad | 🕒 ETA: 26.8 mi

In [ ]:
# Display LangChain results and save
if not final_dataset.empty:
    print(f"\nSuccessfully processed {len(final_dataset)} ads using LangChain!")
    print("\nSample of LangChain enriched data:")
    print(final_dataset.head())
    
    print(f"\nColumns match template: {list(final_dataset.columns) == list(template.columns)}")
    print(f"Template columns: {list(template.columns)}")
    print(f"LangChain enriched columns: {list(final_dataset.columns)}")
    
    # Show data quality improvements
    print(f"\nData Quality Check:")
    print(f"Headlines generated: {len([h for h in final_dataset['headline_text'] if h])}")
    print(f"Categories assigned: {final_dataset['category'].nunique()} unique categories")
    print(f"Platforms suggested: {final_dataset['platform'].value_counts().to_dict()}")
    
else:
    print("No ads were successfully processed. Please check Ollama connection and model availability.")

## Scale up processing and save results

Once you're satisfied with the test batch, you can process more ads and save the results:

In [ ]:
# Update file paths to use new pipeline directory structure
import os

# Update save locations to use pipeline directory
final_file = '../data/pipeline/enriched_ads_7k_complete.csv'

print(f"Files will be saved to: {final_file}")
print("Updated to use data/pipeline directory structure!")